# Global → local spatial autocorrelation

Global Moran's I ranking (validated against the from-scratch `moran_scratch`), then LISA + Gi\* on the top-ranked genes, FDR within a gene, isolate masking, and hotspot overlays on the aligned H&E with the compartment-recovery check.

## Smoke test — ranking + scratch parity

The from-scratch Moran's I must match squidpy on the same row-standardized weights before the ranking is trusted.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pathlib, sys
import numpy as np

# Offline smoke data: the project's committed synthetic fixture (no download).
_root = pathlib.Path.cwd()
_root = _root if (_root / "pyproject.toml").exists() else _root.parent
sys.path.insert(0, str(_root / "tests" / "fixtures"))
from synthetic import make_synthetic_visium

from visium_spatial.build_graph import build_spatial_graph, to_libpysal_weights
from visium_spatial.global_autocorr import rank_svgs, top_genes
from visium_spatial.moran_scratch import morans_i

adata = make_synthetic_visium(seed=0)
build_spatial_graph(adata)
ranked = rank_svgs(adata, seed=0)

W = to_libpysal_weights(adata, transform="r")
W_dense = W.full()[0]
for g in adata.var_names:
    x = np.asarray(adata[:, g].X).ravel().astype(float)
    assert np.isclose(ranked.loc[g, "I"], morans_i(x, W_dense), atol=1e-6)
print("scratch <-> squidpy parity: OK")
ranked

## Local indicators + hotspot overlays

LISA (HH/LH/LL/HL) and Gi\* on the top-ranked genes, with per-spot FDR within a gene and isolate masking, drawn over the tissue via the scalefactor. On a real section pass the hires H&E array as `image=` and get `SCALEF` from `extract_scalefactors`.

In [ ]:
import matplotlib.pyplot as plt
from visium_spatial.local_autocorr import local_moran, getis_ord_gi, lisa_quadrants
from visium_spatial.multitest import fdr_within_gene, mask_isolates
from visium_spatial.overlay import plot_hotspots

top = top_genes(ranked, n=2, max_pval=0.05)
SCALEF = 0.15  # tissue_hires_scalef; from extract_scalefactors on a real section

for g in top.index:
    x = np.asarray(adata[:, g].X).ravel().astype(float)
    lm = local_moran(x, W, seed=0)
    _, q = fdr_within_gene(mask_isolates(lm.p_sim, W))   # isolates -> NaN -> excluded
    labels = lisa_quadrants(lm, p_thresh=0.05, pvals=q)
    gi = getis_ord_gi(x, W, seed=0)

    fig, (axL, axG) = plt.subplots(1, 2, figsize=(10, 4))
    plot_hotspots(adata, labels, scalefactor=SCALEF, ax=axL, title=f"{g} — LISA")
    plot_hotspots(adata, gi.Zs, scalefactor=SCALEF, ax=axG, title=f"{g} — Gi* z")
    plt.show()

# compartment-recovery check: the planted patch should come back High-High
lm_hh = local_moran(np.asarray(adata[:, "GENE_HH"].X).ravel().astype(float), W, seed=0)
hh_labels = lisa_quadrants(lm_hh, p_thresh=0.05)
print("GENE_HH High-High spots:", int((hh_labels == "HH").sum()))

## Compartment-recovery validation (real section)

No ground-truth compartment annotation ships with the section, so *recovery* is assessed as **concordance**: markers of the same compartment should share High-High LISA hotspots; markers of different compartments should be spatially segregated. Canonical markers (sources below):

| Compartment | Markers |
|---|---|
| B-cell follicle | `MS4A1` (CD20), `CD79A`, `CR2` (CD21/FDC), `CXCL13` (follicle chemokine) |
| Germinal center | `BCL6`, `AICDA` (AID), `RGS13` |
| T-zone / paracortex | `CD3D`, `CD3E`, `CCL19`, `CCL21` (T-zone chemokines) |

Sources: lymphoid chemokine organization ([J. Immunol. 175:4904](https://academic.oup.com/jimmunol/article-abstract/175/8/4904/8036082)); [CXCL13 overview](https://www.sciencedirect.com/topics/immunology-and-microbiology/cxcl13); GC program ([Front. Immunol. 2018](https://pmc.ncbi.nlm.nih.gov/articles/PMC6134015/); [RGS13, PLoS ONE](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0060139)); [CD21/CR2 FDC marker](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6610797/). Runs only if the section is cached via `load_visium_dataset('V1_Human_Lymph_Node')`.

In [ ]:
import numpy as np, scipy.sparse as sp
import matplotlib.pyplot as plt
from pathlib import Path
from scanpy import settings

from visium_spatial.load_visium import load_visium_dataset, extract_scalefactors
from visium_spatial.qc import compute_qc_metrics, filter_spots, filter_genes
from visium_spatial.preprocess import normalize
from visium_spatial.build_graph import build_spatial_graph, to_libpysal_weights
from visium_spatial.local_autocorr import local_moran, lisa_quadrants
from visium_spatial.multitest import fdr_within_gene, mask_isolates
from visium_spatial.overlay import plot_hotspots
from visium_spatial.compartments import hh_spot_set, concordance_matrix

COMPARTMENTS = {
    "B/follicle": ["MS4A1", "CD79A", "CR2", "CXCL13"],
    "germinal_center": ["BCL6", "AICDA", "RGS13"],
    "T-zone": ["CD3D", "CD3E", "CCL19", "CCL21"],
}

def gene_vector(ad, g):
    v = ad[:, g].X                       # real X is sparse; synthetic is dense
    return (v.toarray() if sp.issparse(v) else np.asarray(v)).ravel().astype(float)

if not (Path(settings.datasetdir) / "visium" / "V1_Human_Lymph_Node").exists():
    print("Cache the section first:  load_visium_dataset('V1_Human_Lymph_Node')  (~40 MB).")
else:
    ad = load_visium_dataset("V1_Human_Lymph_Node")
    compute_qc_metrics(ad); ad = filter_spots(ad); filter_genes(ad); normalize(ad)
    build_spatial_graph(ad)
    W = to_libpysal_weights(ad, transform="r")

    g2c = {g: c for c, gs in COMPARTMENTS.items() for g in gs}
    hh_sets = {g: hh_spot_set(local_moran(gene_vector(ad, g), W, seed=0), W) for g in g2c}

    names, M = concordance_matrix(hh_sets)
    pairs = [(i, j) for i in range(len(names)) for j in range(i + 1, len(names))]
    within = [M[i, j] for i, j in pairs if g2c[names[i]] == g2c[names[j]]]
    between = [M[i, j] for i, j in pairs if g2c[names[i]] != g2c[names[j]]]
    print(f"mean Jaccard  within-compartment={np.nanmean(within):.2f}  between={np.nanmean(between):.2f}")

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(M, vmin=0, vmax=1, cmap="magma")
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.set_title("HH-hotspot concordance (Jaccard)"); fig.colorbar(im, ax=ax)
    plt.show()

    # Hotspot overlays on the real H&E: a follicle vs a T-zone marker should sit apart.
    scalef = extract_scalefactors(ad)["tissue_hires_scalef"]
    hires = ad.uns["spatial"][list(ad.uns["spatial"])[0]]["images"]["hires"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for g, ax in zip(["CXCL13", "CCL21"], axes):
        lm = local_moran(gene_vector(ad, g), W, seed=0)
        _, q = fdr_within_gene(mask_isolates(lm.p_sim, W))
        labels = lisa_quadrants(lm, p_thresh=0.05, pvals=q)
        plot_hotspots(ad, labels, scalefactor=scalef, image=hires, ax=ax, title=f"{g} LISA on H&E")
    plt.show()

## Notes

The follicle-vs-paracortex split recovered above is the compartment validation (quantified, not eyeballed). Extensions: add sinus/myeloid markers (`LYZ`, `MARCO`), or overlay Gi\* z instead of LISA labels.